In [ ]:
from setup import setup_and_fetch_variables
environment_variables = setup_and_fetch_variables()
folder_paths = environment_variables.get('folder_paths', [])

In [ ]:
import glob
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import importlib
import get_datetime
importlib.reload(get_datetime)
from get_datetime import (
    # get_media_metadata, 
    plot_counts, extract_datetime_from_mp4
)

import data
importlib.reload(data)
from data import get_media_metadata

In [ ]:
file_paths = glob.glob(os.path.join(folder_paths[0], "*"))
df = get_media_metadata(file_paths)
df.to_csv("output/media_datetime_v3.csv", index=False)

In [ ]:
df

In [ ]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

geolocator = Nominatim(user_agent="color-diary")
reverse = RateLimiter(geolocator.reverse, min_delay_seconds=1)

def get_city(lat, lon):
    if pd.isna(lat) or pd.isna(lon):
        return None
    try:
        location = reverse((lat, lon), language='en')
        if location and "address" in location.raw:
            addr = location.raw["address"]
            return addr.get("city") or addr.get("town") or addr.get("village") or addr.get("state")
    except Exception as e:
        print(f"Reverse geocoding failed for ({lat}, {lon}): {e}")
    return None

df['city'] = df.apply(lambda row: get_city(row['latitude'], row['longitude']) if row['type'] == 'photo' else None, axis=1)

In [ ]:
df

In [ ]:
df.isna().sum()

In [ ]:
df.info()

In [ ]:
plot_counts(df)

In [ ]:
from collections import Counter

def average_rgb(colors):
    colors = [c for c in colors if c and None not in c]
    if not colors:
        return None
    avg = tuple(int(sum(x) / len(x)) for x in zip(*colors))
    return avg

def aggregate_city_colors(df, k=5):
    photo_df = df[df['type'] == 'photo'].copy()
    photo_df = photo_df.dropna(subset=['city', 'color_1', 'color_2', 'color_3'])
    
    # Combine all colors per city
    city_color_data = {}
    for city, group in photo_df.groupby('city'):
        colors = []
        for _, row in group.iterrows():
            for key in ['color_1', 'color_2', 'color_3']:
                if row[key] and None not in row[key]:
                    colors.append(row[key])
        # Use most common or average color clusters
        if not colors:
            continue
        counter = Counter(colors)
        top_colors = [color for color, _ in counter.most_common(k)]
        city_color_data[city] = top_colors
    
    return city_color_data

def plot_color_diary(city_color_data):
    import matplotlib.pyplot as plt

    cities = list(city_color_data.keys())
    n_cities = len(cities)
    
    fig, ax = plt.subplots(figsize=(10, n_cities * 0.5))
    ax.axis('off')

    for i, city in enumerate(cities):
        colors = city_color_data[city]
        for j, color in enumerate(colors):
            rect = plt.Rectangle((j, i), 1, 1, color=[c/255 for c in color])
            ax.add_patch(rect)
        ax.text(-0.5, i + 0.5, city, va='center', ha='right', fontsize=10)

    ax.set_xlim(-1, max(len(colors) for colors in city_color_data.values()))
    ax.set_ylim(0, n_cities)
    plt.title("Color Diary by City", fontsize=14)
    plt.tight_layout()
    plt.show()


city_color_data = aggregate_city_colors(df)
plot_color_diary(city_color_data)
